# Разбиение MicroLens-100K на train / validation / test

Цель этапа — произвести корректное разбиение данных: взаимодействия
пользователей с объектами на обучающую, валидационную и тестовую части.

При разбиении необходимо:

- сохранить временной порядок взаимодействий;
- предотвратить утечку информации из будущего;
- сохранить достаточную историю пользователей в обучающей выборке;
- определить поведение для объектов, отсутствующих в обучающей выборке

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from prompt_toolkit import validation

In [2]:
data_path = Path("../../data/raw/microlens")
interactions_path = data_path / "MicroLens-100k_pairs.csv"
interactions_df = pd.read_csv(interactions_path)
interactions_df.head(10)

,user,item,timestamp
0,36121,9580,1583378629552
1,26572,9580,1583436719018
2,94805,9580,1584083806481
3,37550,9580,1584412681021
4,89825,9580,1584649439020
5,14601,9580,1584848802432
6,15061,9580,1585388171106
7,6364,9580,1585390736041
8,99661,9580,1585392581153
9,57849,9580,1585394954324


In [3]:
interactions_df["datetime"] = pd.to_datetime(interactions_df["timestamp"], unit="ms")

In [4]:
interactions_df = interactions_df.sort_values(["user", "timestamp"]).reset_index(drop=True)
interactions_df.head(10)

,user,item,timestamp,datetime
0,1,1958,1648809472482,2022-04-01 10:37:52.482
1,1,6346,1654693851331,2022-06-08 13:10:51.331
2,1,15223,1658056684232,2022-07-17 11:18:04.232
3,1,17700,1658663077653,2022-07-24 11:44:37.653
4,1,1707,1660215530977,2022-08-11 10:58:50.977
5,2,16657,1653002079551,2022-05-19 23:14:39.551
6,2,915,1654949021395,2022-06-11 12:03:41.395
7,2,9127,1657546790850,2022-07-11 13:39:50.850
8,2,3775,1657722544814,2022-07-13 14:29:04.814
9,2,9500,1657754015460,2022-07-13 23:13:35.460


Посмотрим разделение для 1 пользователя:

In [5]:
user_id = 1
user_history = interactions_df[interactions_df["user"] == user_id]
user_history

,user,item,timestamp,datetime
0,1,1958,1648809472482,2022-04-01 10:37:52.482
1,1,6346,1654693851331,2022-06-08 13:10:51.331
2,1,15223,1658056684232,2022-07-17 11:18:04.232
3,1,17700,1658663077653,2022-07-24 11:44:37.653
4,1,1707,1660215530977,2022-08-11 10:58:50.977


In [6]:
len(user_history)

5

In [7]:
train_user = user_history.iloc[:-2]
validation_user = user_history.iloc[-2:-1]
test_user = user_history.iloc[-1:]

In [8]:
print("train_user: ", train_user)
print("validation_user: ", validation_user)
print("test_user: ", test_user)

train_user:     user   item      timestamp                datetime
0     1   1958  1648809472482 2022-04-01 10:37:52.482
1     1   6346  1654693851331 2022-06-08 13:10:51.331
2     1  15223  1658056684232 2022-07-17 11:18:04.232
validation_user:     user   item      timestamp                datetime
3     1  17700  1658663077653 2022-07-24 11:44:37.653
test_user:     user  item      timestamp                datetime
4     1  1707  1660215530977 2022-08-11 10:58:50.977


Для каждого пользователя взаимодействия предварительно отсортированы в хронологическом порядке.

Используется схема leave-last-two:
- последнее взаимодействие пользователя относится к `test`;
- предпоследнее взаимодействие относится к `validation`;
- все более ранние взаимодействия относятся к `train`.

В MicroLens-100K каждый пользователь имеет не менее 5 взаимодействий, после такого разбиения у каждого пользователя останется не менее 3 взаимодействий в обучающей выборке.

Теперь проведем разбиение для всех пользователей:

In [9]:
interactions_df["position_from_end"] = interactions_df.groupby("user").cumcount(ascending=False)

In [10]:
interactions_df.head(10)

,user,item,timestamp,datetime,position_from_end
0,1,1958,1648809472482,2022-04-01 10:37:52.482,4
1,1,6346,1654693851331,2022-06-08 13:10:51.331,3
2,1,15223,1658056684232,2022-07-17 11:18:04.232,2
3,1,17700,1658663077653,2022-07-24 11:44:37.653,1
4,1,1707,1660215530977,2022-08-11 10:58:50.977,0
5,2,16657,1653002079551,2022-05-19 23:14:39.551,6
6,2,915,1654949021395,2022-06-11 12:03:41.395,5
7,2,9127,1657546790850,2022-07-11 13:39:50.850,4
8,2,3775,1657722544814,2022-07-13 14:29:04.814,3
9,2,9500,1657754015460,2022-07-13 23:13:35.460,2


In [11]:
test_df = interactions_df[interactions_df["position_from_end"] == 0].copy()
validation_df = interactions_df[interactions_df["position_from_end"] == 1].copy()
train_df = interactions_df[interactions_df["position_from_end"] >= 2].copy()

In [12]:
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train: 519405
Validation: 100000
Test: 100000


In [13]:
train_items = set(train_df["item"])
validation_items = set(validation_df["item"])
test_items = set(test_df["item"])

In [14]:
new_validation_items = validation_items - train_items

print(
    "Объектов validation, отсутствующих в train:",
    len(new_validation_items)
)

Объектов validation, отсутствующих в train: 252


In [15]:
new_test_items = test_items - train_items

print(
    "Объектов test, отсутствующих в train:",
    len(new_test_items)
)

Объектов test, отсутствующих в train: 313


In [16]:
validation_cold = validation_df[
    ~validation_df["item"].isin(train_items)
]

print(
    "Validation-взаимодействий с новыми объектами:",
    len(validation_cold)
)

Validation-взаимодействий с новыми объектами: 547


In [17]:
test_cold = test_df[
    ~test_df["item"].isin(train_items)
]

print(
    "Test-взаимодействий с новыми объектами:",
    len(test_cold)
)

Test-взаимодействий с новыми объектами: 1977


### Объекты, отсутствующие в обучающей выборке

После разбиения данных часть объектов из валидационной и тестовой выборок не встречается в обучающей выборке.

Для `validation` обнаружено 252 таких объекта, которым соответствует 547 взаимодействий.
Для `test` обнаружено 313 таких объектов, которым соответствует 1 977 взаимодействий.

Таким образом, 0.55% валидационных и 1.98% тестовых взаимодействий относятся к объектам, по которым в обучающей выборке отсутствует коллаборативный сигнал.

Такие объекты следует рассматривать отдельно от редких объектов, которые имеют небольшое, но ненулевое количество взаимодействий в `train`.

In [18]:
train_item_interactions = train_df.groupby("item").size()
train_item_interactions.describe()

count    19362.000000
mean        26.825999
std         34.578721
min          1.000000
25%          6.000000
50%         15.000000
75%         35.000000
max        546.000000
dtype: float64

In [19]:
train_item_interactions.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

0.10      3.0
0.25      6.0
0.50     15.0
0.75     35.0
0.90     64.0
0.95     87.0
0.99    149.0
dtype: float64

In [20]:
low_items = train_item_interactions[train_item_interactions <= 6]
medium_items = train_item_interactions[(train_item_interactions > 6) & (train_item_interactions <= 15)]
warm_items = train_item_interactions[(train_item_interactions > 15) & (train_item_interactions <= 35)]
hot_items = train_item_interactions[train_item_interactions > 35]

print("Low:", len(low_items))
print("Medium:", len(medium_items))
print("Warm:", len(warm_items))
print("Hot:", len(hot_items))

Low: 5038
Medium: 4694
Warm: 4902
Hot: 4728


In [21]:
n_train_items = train_df["item"].nunique()

print("Объектов в train:", n_train_items)
print("Объектов во всём MicroLens:", interactions_df["item"].nunique())

Объектов в train: 19362
Объектов во всём MicroLens: 19738


In [22]:
items_not_in_train = (set(interactions_df["item"]) - set(train_df["item"]))

print("Объектов, полностью отсутствующих в train:", len(items_not_in_train))

Объектов, полностью отсутствующих в train: 376


На основании количества взаимодействий в обучающей выборке объекты, представленные в `train`, были разделены по квартилям:

- `Low`: 1–6 взаимодействий — 5038 объектов;
- `Medium`: 7–15 взаимодействий — 4694 объекта;
- `Warm`: 16–35 взаимодействий — 4902 объекта;
- `Hot`: более 35 взаимодействий — 4728 объектов.

Дополнительно выделена группа `Unseen` — объекты, полностью отсутствующие
в обучающей выборке. Таких объектов обнаружено 376.

Группы определяются по обучающей выборке, поэтому информация
из `validation` и `test` не используется при оценке силы коллаборативного
сигнала.

In [23]:
validation_df = validation_df.copy()
validation_df["train_item_interactions"] = (validation_df["item"].map(train_item_interactions).fillna(0).astype(int))
validation_df.head()

,user,item,timestamp,datetime,position_from_end,train_item_interactions
3,1,17700,1658663077653,2022-07-24 11:44:37.653,1,26
10,2,10949,1658467984213,2022-07-22 05:33:04.213,1,88
15,3,5408,1660965649542,2022-08-20 03:20:49.542,1,23
20,4,10743,1654408947388,2022-06-05 06:02:27.388,1,47
25,5,8461,1660817250322,2022-08-18 10:07:30.322,1,33


In [24]:
test_df = test_df.copy()
test_df["train_item_interactions"] = (test_df["item"].map(train_item_interactions).fillna(0).astype(int))
test_df.head()

,user,item,timestamp,datetime,position_from_end,train_item_interactions
4,1,1707,1660215530977,2022-08-11 10:58:50.977,0,3
11,2,5299,1659082495634,2022-07-29 08:14:55.634,0,40
16,3,18348,1662071625231,2022-09-01 22:33:45.231,0,36
21,4,11230,1659064899648,2022-07-29 03:21:39.648,0,39
26,5,4365,1662277520276,2022-09-04 07:45:20.276,0,45


In [25]:
def get_item_group(n_interactions):
    if n_interactions == 0:
        return "Unseen"
    elif n_interactions <= 6:
        return "Low"
    elif n_interactions <= 15:
        return "Medium"
    elif n_interactions <= 35:
        return "Warm"
    else:
        return "Hot"

In [26]:
validation_df["item_group"] = (validation_df["train_item_interactions"].apply(get_item_group))
test_df["item_group"] = (test_df["train_item_interactions"].apply(get_item_group))

In [27]:
validation_df[["user", "item", "train_item_interactions", "item_group"]].head(10)

,user,item,train_item_interactions,item_group
3,1,17700,26,Warm
10,2,10949,88,Hot
15,3,5408,23,Warm
20,4,10743,47,Hot
25,5,8461,33,Warm
33,6,764,33,Warm
41,7,6672,48,Hot
46,8,9005,41,Hot
52,9,6311,21,Warm
65,10,12254,171,Hot


In [28]:
validation_df["item_group"].value_counts()

item_group
Hot       51205
Warm      26289
Medium    14165
Low        7794
Unseen      547
Name: count, dtype: int64

In [29]:
test_df["item_group"].value_counts()

item_group
Hot       38232
Warm      26484
Medium    18112
Low       15195
Unseen     1977
Name: count, dtype: int64

In [30]:
train_df = train_df.drop(columns=["position_from_end"])
validation_df = validation_df.drop(columns=["position_from_end"])
test_df = test_df.drop(columns=["position_from_end"])

In [31]:
processed_path = Path("../../data/processed/microlens")
processed_path.mkdir(parents=True, exist_ok=True)

In [32]:
train_df.to_parquet(processed_path / "train.parquet", index=False)
validation_df.to_parquet(processed_path / "validation.parquet", index=False)
test_df.to_parquet(processed_path / "test.parquet", index=False)

In [33]:
train_item_stats = (train_item_interactions.reset_index(name="n_interactions"))
train_item_stats.head()

,item,n_interactions
0,1,2
1,2,122
2,3,57
3,4,29
4,5,18


# Итоги разбиения MicroLens-100K

Для формирования обучающей, валидационной и тестовой выборок использована схема `leave-last-two`.

Для каждого пользователя:

- последнее взаимодействие помещается в `test`;
- предпоследнее — в `validation`;
- остальные взаимодействия — в `train`.

В результате получены:

- `train` — 519 405 взаимодействий;
- `validation` — 100 000 взаимодействий;
- `test` — 100 000 взаимодействий.

Все 100 000 пользователей представлены во всех трёх выборках.
Минимальная длина пользовательской истории в `train` составляет
3 взаимодействия.

Временной порядок взаимодействий сохраняется, поэтому для каждого пользователя
обучающие события предшествуют валидационному и тестовому событиям.

В validation и test присутствуют объекты, отсутствующие в обучающей выборке.
Такие объекты выделяются в отдельную группу `Unseen`.

Для объектов, присутствующих в `train`, сила коллаборативного сигнала
определяется по количеству обучающих взаимодействий:

- `Low`: 1–6;
- `Medium`: 7–15;
- `Warm`: 16–35;
- `Hot`: более 35 взаимодействий.

Границы определены по квартилям распределения популярности объектов
в обучающей выборке.